# Qwen-Image 2.1 image generation with OpenVINO

Qwen-Image 2.1 is a unified image generation model that supports text-to-image generation and image-conditioned editing in one pipeline. The prompt and condition images are encoded together by Qwen3-VL and processed by a block-causal diffusion transformer.

This tutorial demonstrates how to:

- download and convert [Qwen-Image 2.1](https://huggingface.co/Qwen/Qwen-Image-2.1) to OpenVINO IR;
- run text-to-image generation with OpenVINO GenAI;
- run image-conditioned editing with the same exported model;
- measure pipeline loading, first-run, and warm-run latency;
- launch an interactive demo.

> **Important:** Qwen-Image 2.1 image conditioning is not classic image-to-image generation based on adding noise to an initial image. The condition image is part of the multimodal context, so this notebook intentionally does not expose a `strength` parameter.

⚠️ **EXPERIMENTAL NOTEBOOK**

This notebook demonstrates a model that has not been fully validated with OpenVINO. It may be fully supported and validated in the future.

> **Resource note:** Conversion and inference may require substantial RAM and disk space. Start with FP16 and a 1024 x 1024 output; enable compression only after validating model quality.

#### Table of contents

- [Prerequisites](#Prerequisites)
  - [Installation](#Installation)
- [Select model and export options](#Select-model-and-export-options)
- [Convert the model to OpenVINO IR](#Convert-the-model-to-OpenVINO-IR)
- [Run OpenVINO GenAI inference](#Run-OpenVINO-GenAI-inference)
  - [Text-to-image](#Text-to-image)
  - [Image-conditioned editing](#Image-conditioned-editing)
- [Benchmark](#Benchmark)
- [Interactive demo](#Interactive-demo)


## Prerequisites

This notebook downloads [`Qwen/Qwen-Image-2.1`](https://huggingface.co/Qwen/Qwen-Image-2.1) from the Hugging Face Hub during conversion.

The installation uses:

1. the merged Qwen-Image 2.1 support from the Optimum Intel `main` branch;
2. OpenVINO, OpenVINO Tokenizers, and OpenVINO GenAI nightly wheels;
3. the Diffusers `main` branch containing Qwen-Image 2.1 support.


In [ ]:
import os
import requests
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display

for fname in ("notebook_utils.py", "cmd_helper.py", "pip_helper.py"):
    if not Path(fname).exists():
        response = requests.get(
            f"https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/{fname}",
            timeout=30,
        )
        response.raise_for_status()
        Path(fname).write_text(response.text, encoding="utf-8")

from pip_helper import pip_install

pip_install(
    "-q",
    "gradio>=4.19,<6",
    "torch>=2.4",
    "accelerate",
    "nncf>=2.15.0",
    "numpy",
    "pillow",
    "ipywidgets",
    "tqdm",
    "--extra-index-url",
    "https://download.pytorch.org/whl/cpu",
)

# Read more about telemetry collection at
# https://github.com/openvinotoolkit/openvino_notebooks#telemetry
from notebook_utils import collect_telemetry

collect_telemetry("qwen-image-2.1.ipynb")

### Installation

Install the required dependencies:

- Optimum Intel and Diffusers are installed from their upstream `main` branches.
- OpenVINO, OpenVINO Tokenizers, and OpenVINO GenAI are installed from the OpenVINO nightly wheel index.

Restart the notebook kernel after installing or changing these packages.


In [ ]:
pip_install(
    "-qU",
    "git+https://github.com/huggingface/diffusers.git@main",
    "git+https://github.com/huggingface/optimum-intel.git@main",
    "--extra-index-url",
    "https://download.pytorch.org/whl/cpu",
)
pip_install(
    "-qU",
    "--pre",
    "openvino",
    "openvino-tokenizers",
    "openvino-genai",
    "--extra-index-url",
    "https://storage.openvinotoolkit.org/simple/wheels/nightly",
)

print("Packages installed. Restart the notebook kernel before continuing.")

## Select model and export options

The source model is [`Qwen/Qwen-Image-2.1`](https://huggingface.co/Qwen/Qwen-Image-2.1). The conversion downloads model files through the Hugging Face Hub cache and writes the OpenVINO model to a separate output directory.

`FP16` is the default until compressed-model quality has been evaluated. Selecting `INT8` or `INT4` enables Optimum Intel weight compression during export.


In [ ]:
model_id = "Qwen/Qwen-Image-2.1"

export_root = widgets.Text(
    value=os.getenv("QWEN_IMAGE_21_OV_DIR", "qwen-image-2.1-ov"),
    description="Export folder:",
    layout=widgets.Layout(width="90%"),
    style={"description_width": "120px"},
)
weight_format = widgets.Dropdown(
    options=("FP16", "INT8", "INT4"),
    value="FP16",
    description="Weights:",
    style={"description_width": "120px"},
)

print(f"Source model: https://huggingface.co/{model_id}")
display(export_root, weight_format)

In [ ]:
export_base_dir = Path(export_root.value).expanduser()
model_dir = export_base_dir / weight_format.value

additional_args = {
    "task": "text-to-image",
    "weight-format": weight_format.value.lower(),
}
if weight_format.value == "INT4":
    additional_args.update({"group-size-fallback": "ignore"})

print(f"Hugging Face source model: {model_id}")
print(f"OpenVINO output: {model_dir.resolve()}")

## Convert the model to OpenVINO IR

Optimum CLI downloads the source model from the Hugging Face Hub and exports it to the selected local directory. The conversion is skipped only when the completion marker matches the selected model and weight format.

Delete or rename the target directory to force a clean reconversion after updating Optimum Intel or the nightly OpenVINO packages.


In [ ]:
from cmd_helper import optimum_cli

export_complete_marker = model_dir / ".openvino_export_complete"
export_signature = f"model_id={model_id}\nweight_format={weight_format.value}\n"

if export_complete_marker.is_file():
    if export_complete_marker.read_text(encoding="utf-8") != export_signature:
        raise RuntimeError(
            f"The export in {model_dir.resolve()} was created from another source or configuration. "
            "Remove it or select another export folder before retrying."
        )
    print(f"Using existing OpenVINO model from {model_dir.resolve()}")
elif model_dir.exists():
    raise RuntimeError(
        f"The export directory {model_dir.resolve()} exists but has no completion marker. "
        "It is probably left from an interrupted export. Remove it or select another export folder before retrying."
    )
else:
    optimum_cli(model_id, model_dir, additional_args=additional_args)
    export_complete_marker.write_text(export_signature, encoding="utf-8")

## Run OpenVINO GenAI inference

The exported directory is used by both OpenVINO GenAI pipelines. Qwen-Image 2.1 defaults are used throughout the notebook:

- 40 denoising steps;
- guidance scale 1.0;
- no negative prompt;
- 1024 x 1024 output;
- fixed random seed.

The exact constructor or `generate` signature may change while the OpenVINO GenAI integration is under development. If that happens, only the two pipeline-loading/generation cells and `gradio_helper.py` should need adjustment.


In [ ]:
import gc
import time
from datetime import datetime

import numpy as np
import openvino as ov
import openvino_genai as ov_genai
from PIL import Image, ImageOps
from tqdm.auto import tqdm

from notebook_utils import device_widget


def image_to_tensor(image: Image.Image) -> ov.Tensor:
    image_data = np.asarray(image.convert("RGB"), dtype=np.uint8)[None]
    return ov.Tensor(image_data)


def output_to_image(output) -> Image.Image:
    data = output.data if hasattr(output, "data") else output
    array = np.asarray(data)
    if array.ndim == 4:
        array = array[0]
    return Image.fromarray(array).convert("RGB")


def make_image_comparison(
    source_image: Image.Image, generated_image: Image.Image
) -> Image.Image:
    source_preview = ImageOps.fit(
        source_image, generated_image.size, method=Image.Resampling.LANCZOS
    )
    comparison = Image.new("RGB", (generated_image.width * 2, generated_image.height))
    comparison.paste(source_preview, (0, 0))
    comparison.paste(generated_image, (generated_image.width, 0))
    return comparison


def save_generated_image(
    image: Image.Image, scenario: str, precision: str, inference_device: str, seed: int
) -> Path:
    output_dir = Path("generated_images")
    output_dir.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d-%H%M%S-%f")
    safe_device = inference_device.replace(":", "-").replace(".", "-")
    output_path = (
        output_dir
        / f"qwen-image-2.1-{scenario}-{precision.lower()}-{safe_device.lower()}-seed{seed}-{timestamp}.png"
    )
    image.save(output_path)
    print(f"Saved image to {output_path.resolve()}")
    return output_path


def generate_with_progress(
    pipeline, *args, num_inference_steps: int, description: str, **kwargs
):
    progress = tqdm(total=num_inference_steps, desc=description)

    def callback(step, num_steps, latent):
        completed_steps = min(step + 1, num_steps)
        progress.update(max(0, completed_steps - progress.n))
        return False

    try:
        return pipeline.generate(
            *args,
            num_inference_steps=num_inference_steps,
            callback=callback,
            **kwargs,
        )
    finally:
        progress.close()


def print_perf_metrics(output):
    metrics = getattr(output, "perf_metrics", None)
    if metrics is None:
        print(
            "This OpenVINO GenAI image output does not expose perf_metrics; wall-clock metrics are used."
        )
        return

    metric_methods = {
        "Load time": "get_load_time",
        "Generate duration": "get_generate_duration",
        "Transformer duration": "get_transformer_infer_duration",
        "VAE encoder duration": "get_vae_encoder_infer_duration",
        "VAE decoder duration": "get_vae_decoder_infer_duration",
    }
    for label, method_name in metric_methods.items():
        method = getattr(metrics, method_name, None)
        if method is not None:
            print(f"{label}: {method()}")


device = device_widget(default="CPU", exclude=["NPU"])
device

### Text-to-image

Load the text-to-image pipeline and record model loading time separately from generation latency.


In [ ]:
t2i_load_started = time.perf_counter()
t2i_pipe = ov_genai.Text2ImagePipeline(str(model_dir), device.value)
t2i_load_seconds = time.perf_counter() - t2i_load_started

print(f"Text-to-image pipeline load time: {t2i_load_seconds:.2f} s")

In [ ]:
t2i_prompt = 'A cozy coffee shop with a chalkboard sign reading "Qwen Coffee", cinematic lighting'
t2i_num_inference_steps = 40
t2i_seed = 42

t2i_first_started = time.perf_counter()
t2i_output = generate_with_progress(
    t2i_pipe,
    t2i_prompt,
    height=1024,
    width=1024,
    guidance_scale=1.0,
    num_inference_steps=t2i_num_inference_steps,
    description="Text-to-image",
    generator=ov_genai.TorchGenerator(t2i_seed),
)
t2i_first_run_seconds = time.perf_counter() - t2i_first_started

print(f"First text-to-image generation: {t2i_first_run_seconds:.2f} s")
print_perf_metrics(t2i_output)
t2i_image = output_to_image(t2i_output)
t2i_output_path = save_generated_image(
    t2i_image, "t2i", weight_format.value, device.value, t2i_seed
)
t2i_image

#### Release the text-to-image pipeline

Release the compiled text-to-image pipeline before loading the image editing pipeline.


In [ ]:
if "t2i_pipe" in globals():
    del t2i_pipe
    gc.collect()
    print("Text-to-image pipeline released.")
else:
    print("Text-to-image pipeline is not loaded.")

### Image-conditioned editing

Select a local condition image and describe the requested edit. The image is multimodal context for Qwen-Image 2.1; it is not a noisy initialization controlled by `strength`.

The current scaffold uses `Image2ImagePipeline` and passes the condition tensor as the second positional argument, matching the existing OpenVINO GenAI image API. Height and width are intentionally omitted: the Qwen-Image 2.1 pipeline derives a 32-aligned output resolution near 1024² pixels from the condition image aspect ratio. Keep this call isolated until the final Qwen-Image 2.1 GenAI API is confirmed.


In [ ]:
sample_image_url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/cat.png"
sample_image_path = Path("sample_cat.png")
if not sample_image_path.is_file():
    response = requests.get(sample_image_url, timeout=30)
    response.raise_for_status()
    sample_image_path.write_bytes(response.content)

input_image_path = widgets.Text(
    value=os.getenv("QWEN_IMAGE_21_INPUT_IMAGE", str(sample_image_path)),
    description="Input image:",
    placeholder="Path to a local image used for editing",
    layout=widgets.Layout(width="90%"),
    style={"description_width": "120px"},
)
input_image_path

In [ ]:
condition_image_path = Path(input_image_path.value).expanduser()
if not condition_image_path.is_file():
    raise FileNotFoundError(
        "Set QWEN_IMAGE_21_INPUT_IMAGE or the Input image widget to a local image file."
    )

condition_image = Image.open(condition_image_path).convert("RGB")
condition_tensor = image_to_tensor(condition_image)
edit_prompt = "Put a tiny red top hat and a blue bow tie on the cat. Preserve the cat's identity, face, pose, and the original composition."
i2i_num_inference_steps = 40
i2i_seed = 42

i2i_load_started = time.perf_counter()
i2i_pipe = ov_genai.Image2ImagePipeline(str(model_dir), device.value)
i2i_load_seconds = time.perf_counter() - i2i_load_started

i2i_first_started = time.perf_counter()
i2i_output = generate_with_progress(
    i2i_pipe,
    edit_prompt,
    condition_tensor,
    guidance_scale=1.0,
    num_inference_steps=i2i_num_inference_steps,
    description="Image editing",
    generator=ov_genai.TorchGenerator(i2i_seed),
)
i2i_first_run_seconds = time.perf_counter() - i2i_first_started

print(f"Image editing pipeline load time: {i2i_load_seconds:.2f} s")
print(f"First image editing generation: {i2i_first_run_seconds:.2f} s")
print_perf_metrics(i2i_output)
i2i_image = output_to_image(i2i_output)
i2i_output_path = save_generated_image(
    i2i_image, "editing", weight_format.value, device.value, i2i_seed
)
i2i_comparison = make_image_comparison(condition_image, i2i_image)
i2i_comparison

#### Release the image editing pipeline

Release the compiled image editing pipeline before running benchmarks or launching the interactive demo.


In [ ]:
if "i2i_pipe" in globals():
    del i2i_pipe
    gc.collect()
    print("Image editing pipeline released.")
else:
    print("Image editing pipeline is not loaded.")

## Benchmark

The benchmark reports end-to-end pipeline latency rather than isolated OpenVINO IR component performance.

- Pipeline loading and the first generation are reported from the preceding cells.
- One additional warm-up generation is excluded.
- Warm latency reports minimum, median, and mean over the selected number of runs.
- T2I and image-conditioned editing are loaded, measured, and released sequentially.

For reproducible comparisons, keep the device, precision, resolution, number of steps, guidance scale, prompt, condition image, and KV-cache behavior unchanged. Close other resource-intensive applications before collecting final numbers.

Run the next cell, select the number of measured warm runs, and then run the benchmark cell below it.


In [ ]:
benchmark_runs = widgets.IntSlider(
    value=3,
    min=1,
    max=10,
    step=1,
    description="Warm runs:",
    style={"description_width": "100px"},
)
benchmark_runs

In [ ]:
from statistics import mean, median


def benchmark_generation(generate, runs: int, warmup_runs: int = 1):
    for _ in range(warmup_runs):
        generate()

    durations = []
    for _ in range(runs):
        started = time.perf_counter()
        generate()
        durations.append(time.perf_counter() - started)

    return {
        "minimum": min(durations),
        "median": median(durations),
        "mean": mean(durations),
    }


def generate_t2i_benchmark(pipeline):
    return pipeline.generate(
        t2i_prompt,
        height=1024,
        width=1024,
        guidance_scale=1.0,
        num_inference_steps=40,
        generator=ov_genai.TorchGenerator(t2i_seed),
    )


def generate_i2i_benchmark(pipeline):
    return pipeline.generate(
        edit_prompt,
        condition_tensor,
        guidance_scale=1.0,
        num_inference_steps=40,
        generator=ov_genai.TorchGenerator(i2i_seed),
    )


t2i_benchmark_pipe = ov_genai.Text2ImagePipeline(str(model_dir), device.value)
try:
    t2i_warm = benchmark_generation(
        lambda: generate_t2i_benchmark(t2i_benchmark_pipe), benchmark_runs.value
    )
finally:
    del t2i_benchmark_pipe
    gc.collect()

i2i_benchmark_pipe = ov_genai.Image2ImagePipeline(str(model_dir), device.value)
try:
    i2i_warm = benchmark_generation(
        lambda: generate_i2i_benchmark(i2i_benchmark_pipe), benchmark_runs.value
    )
finally:
    del i2i_benchmark_pipe
    gc.collect()

benchmark_results = {
    "Text-to-image": {
        "load": t2i_load_seconds,
        "first": t2i_first_run_seconds,
        **t2i_warm,
    },
    "Image editing": {
        "load": i2i_load_seconds,
        "first": i2i_first_run_seconds,
        **i2i_warm,
    },
}

print(
    f"{'Scenario':<18} {'Load (s)':>10} {'First (s)':>11} {'Min (s)':>10} {'Median (s)':>12} {'Mean (s)':>11}"
)
for scenario, values in benchmark_results.items():
    print(
        f"{scenario:<18} {values['load']:>10.2f} {values['first']:>11.2f} "
        f"{values['minimum']:>10.2f} {values['median']:>12.2f} {values['mean']:>11.2f}"
    )

## Interactive demo

The demo allows selecting Text-to-Image or Image Editing, FP16/INT8/INT4 precision, and an available OpenVINO device. It keeps only the selected pipeline in memory and releases it when the configuration changes.


In [ ]:
from gradio_helper import make_demo

for pipeline_name in ("t2i_pipe", "i2i_pipe"):
    if pipeline_name in globals():
        del globals()[pipeline_name]
gc.collect()

demo = make_demo(
    export_base_dir,
    default_precision=weight_format.value,
    default_device=device.value,
    editing_sample=sample_image_path,
)
demo.launch(debug=False)